# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okashaahmed2/Flyrankaiintern/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [7]:
#First load the model
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

url = "https://raw.githubusercontent.com/okashaahmed2/Flyrankaiintern/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['target'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = ['impressions_90d', 'ctr', 'avg_position']
X, y = df[feature_cols], df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print('Model reloaded — same 3 features, same split as w05_model.ipynb.')

Model reloaded — same 3 features, same split as w05_model.ipynb.


**Priority score:** `decline_probability × impressions_90d`. A page the model is only mildly sure about but that carries a lot of traffic outranks a page the model is very sure about but that barely gets seen — this is the same impact-weighting used in the capstone.

**Reason code:** rather than showing a bare number, each page's own `avg_position`, `ctr`, and `impressions_90d` are compared against the *training-set median* for that feature. If a page's position is worse than typical, or its CTR is below typical, that's stated in plain words — this is what makes the queue something an editor can trust and argue with, not a black-box score.

In [8]:
proba = rf.predict_proba(X_test)[:, 1]

queue = X_test.copy()
queue['decline_probability'] = proba
queue['priority_score'] = proba * queue['impressions_90d']

med_impr = X_train['impressions_90d'].median()
med_ctr = X_train['ctr'].median()
med_pos = X_train['avg_position'].median()

def reason_code(row):
    reasons = []
    if row['avg_position'] > med_pos:
        reasons.append('position worse than typical')
    if row['ctr'] < med_ctr:
        reasons.append('CTR below typical')
    if row['impressions_90d'] > med_impr:
        reasons.append('high-traffic page, worth prioritizing')
    return '; '.join(reasons) if reasons else 'borderline — flagged mainly on model probability'

queue['reason_code'] = queue.apply(reason_code, axis=1)
ranked_queue = queue.sort_values('priority_score', ascending=False)

print('=== TOP 15 PAGES — RANKED ACTION QUEUE ===')
print(ranked_queue[['impressions_90d', 'ctr', 'avg_position', 'decline_probability', 'priority_score', 'reason_code']].head(15).round(3).to_string())

=== TOP 15 PAGES — RANKED ACTION QUEUE ===
       impressions_90d   ctr  avg_position  decline_probability  priority_score                                                                            reason_code
6653            517715  0.14           4.2                 0.47       243326.05                                                  high-traffic page, worth prioritizing
21819           463103  0.41           2.3                 0.41       189872.23                                                  high-traffic page, worth prioritizing
27478           236803  0.26           4.4                 0.50       118401.50                                                  high-traffic page, worth prioritizing
22028           213963  0.10           4.7                 0.55       117679.65                                                  high-traffic page, worth prioritizing
3603            130932  0.04          40.1                 0.85       111292.20  position worse than typical; CTR below ty

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a weekly triage list for a content editor — not an auto-publisher. The editor opens the top 15–20 rows, reads the reason code, and decides whether to refresh, expand, or leave a page alone.

**Where it stops being valid:**
- Trained on one mid-panel snapshot (March 2026) of the small anonymized sample — it has not been validated on the full ~79M-row release or on a different time window.
- Only 3 features (`impressions_90d`, `ctr`, `avg_position`) — it does not see page speed, backlinks, content quality, or recent manual edits, so it cannot explain *why* a page dropped, only *that* it looks like it did.
- Built for this lane's specific label (`trend_direction == 'down'`) — it is not a general-purpose SEO score and should not be reused for a different question without re-checking the data contract.

In [9]:
print('Scope check — this model is valid only for:')
print(f"  Feature set : {feature_cols}")
print(f"  Snapshot    : March 2026, {len(df):,} anonymized pages")
print(f"  Pages scored in this run: {len(X_test):,}")

Scope check — this model is valid only for:
  Feature set : ['impressions_90d', 'ctr', 'avg_position']
  Snapshot    : March 2026, 30,000 anonymized pages
  Pages scored in this run: 6,000


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged page, a person checks:**
- Was this page recently and deliberately changed (redesign, migration) — could that explain the metric shift instead of organic decline?
- Is the traffic volume behind this flag large enough to trust, or is it thin enough that normal week-to-week noise could produce the same pattern?
- Does the reason code actually make sense for this specific page, or does it look like an edge case the median-comparison heuristic doesn't handle well?

**Never automated — the no-go list:**
- No auto-publishing or auto-editing content directly from this score.
- No auto-deleting, unpublishing, or redirecting a page based on the model alone.
- No treating a single flagged page as proof of *why* it declined — that always needs a human look.

In [10]:
# Pages in the bottom 10% of impressions are the noisiest — flag them for extra manual caution
low_signal_cutoff = queue['impressions_90d'].quantile(0.10)
low_signal_pages = ranked_queue[ranked_queue['impressions_90d'] < low_signal_cutoff]
print(f"Pages below the 10th percentile of impressions (thin signal — needs manual check before acting): {len(low_signal_pages):,}")

Pages below the 10th percentile of impressions (thin signal — needs manual check before acting): 576


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Retrain if any of these happen:**
- Measured F1 on a fresh month's data drops meaningfully below this run's baseline — a real-world performance check, not just a training-time number.
- The declining-page base rate drifts far from the 54.21% seen here — the world the model learned no longer matches the world it's scoring.
- Editors start reporting that reason codes feel wrong or unhelpful often — a qualitative signal that's just as valid as a metric drop.

In [11]:
y_pred = rf.predict(X_test)
current_f1 = f1_score(y_test, y_pred)
current_base_rate = y.mean()

print(f"Retrain-trigger baseline — F1: {current_f1:.4f} | declining-page base rate: {current_base_rate:.2%}")
print("Retrain if a fresh month's F1 falls more than ~10% relative to this baseline,")
print("or if the base rate moves more than ~10 percentage points away from it.")

Retrain-trigger baseline — F1: 0.6383 | declining-page base rate: 54.21%
Retrain if a fresh month's F1 falls more than ~10% relative to this baseline,
or if the base rate moves more than ~10 percentage points away from it.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked queue is saved as a CSV so the capstone paper's "Ranked recommendations" section reads from a real, reproducible file rather than numbers typed in by hand.

In [12]:
import os

os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/refresh_priority_queue.csv', index=False)

print('Exported: work/outputs/refresh_priority_queue.csv')
print(f'Rows exported: {len(ranked_queue):,}')

Exported: work/outputs/refresh_priority_queue.csv
Rows exported: 6,000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.